# 05 Review Overview Dashboard

This notebook reads the deduplicated 9,222 NSFC project records from `data/NSFC正式增量采集_2014-2026_去重筛选最终结果.csv`, uses the new master table with `award_year` from 2010 to 2023 to generate 10 overview metrics, and exports the Fig. 4 dashboard, metric table, and draft results paragraph.

Note: all visible figure text is in English; code comments document the reproducible rules for each step.

In [ ]:
# Import core libraries: pandas handles table preparation, and matplotlib/seaborn export static publication figures.
from pathlib import Path
import math
import re
import textwrap

import numpy as np
import pandas as pd
import matplotlib as mpl
from matplotlib import font_manager
import matplotlib.pyplot as plt

# Register and require Times New Roman; stop immediately if the font is unavailable instead of using a fallback.
TIMES_NEW_ROMAN_PATHS = [
    Path("/Library/Fonts/Times New Roman.ttf"),
    Path("/Library/Fonts/Times New Roman Bold.ttf"),
    Path("/Library/Fonts/Times New Roman Italic.ttf"),
    Path("/Library/Fonts/Times New Roman Bold Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold Italic.ttf"),
]
for font_path in TIMES_NEW_ROMAN_PATHS:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))
try:
    font_manager.findfont("Times New Roman", fallback_to_default=False)
except ValueError as exc:
    raise RuntimeError("Times New Roman is required for figure export but was not found by matplotlib.") from exc

import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch

# Set project paths so inputs and outputs are reproducible when the notebook runs from the project root or code/ directory.
SOURCE_DATA_RELATIVE = Path("data") / "NSFC正式增量采集_2014-2026_去重筛选最终结果.csv"
EXPECTED_RECORDS = 9222
EXPECTED_AWARD_YEAR_RANGE = (2010, 2023)

def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if (candidate / SOURCE_DATA_RELATIVE).exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Cannot locate {SOURCE_DATA_RELATIVE} from the current working directory.")


ROOT = find_project_root()
DATA_PATH = ROOT / SOURCE_DATA_RELATIVE
FIGURE_DIR = ROOT / "output" / "figures"
TABLE_DIR = ROOT / "output" / "tables"
LOG_DIR = ROOT / "output" / "logs"
NOTEBOOK_PATH = ROOT / "code" / "05_review_overview_dashboard.ipynb"

FIGURE_BASE = FIGURE_DIR / "Fig4_overview_dashboard"
TABLE_PATH = TABLE_DIR / "05_dashboard_metrics.csv"
LOG_PATH = LOG_DIR / "05_dashboard_text.md"

# Create only the output directories needed for this task; do not modify data or other subagents' files.
for path in [FIGURE_DIR, TABLE_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Input data: {DATA_PATH}")
print(f"Figure base: {FIGURE_BASE}")

In [ ]:
# Read the master dataset and run minimal integrity checks.
# This notebook does not write back to data; it only maps raw fields to dashboard categories.
df = pd.read_csv(DATA_PATH)

expected_columns = {
    "country", "source_system", "agency_program", "amount_original", "currency",
    "knowledge_prod_place", "research_object_place", "beneficiary_city",
    "matched_method_terms", "matched_object_terms", "matched_performance_terms",
    "project_title", "abstract_text", "keywords_raw", "outcomes_text",
    "include_reason", "review_flag", "award_year"
}
missing_columns = sorted(expected_columns.difference(df.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

N_RECORDS = len(df)
if N_RECORDS != EXPECTED_RECORDS:
    raise ValueError(f"Expected {EXPECTED_RECORDS:,} records in the new master table, got {N_RECORDS:,}.")

df["award_year"] = pd.to_numeric(df["award_year"], errors="raise").astype("Int64")
AWARD_YEAR_MIN = int(df["award_year"].min())
AWARD_YEAR_MAX = int(df["award_year"].max())
if (AWARD_YEAR_MIN, AWARD_YEAR_MAX) != EXPECTED_AWARD_YEAR_RANGE:
    raise ValueError(
        f"Expected award_year range {EXPECTED_AWARD_YEAR_RANGE[0]}-{EXPECTED_AWARD_YEAR_RANGE[1]}, "
        f"got {AWARD_YEAR_MIN}-{AWARD_YEAR_MAX}."
    )

print(f"Rows: {N_RECORDS:,}")
print(f"Award years: {AWARD_YEAR_MIN}-{AWARD_YEAR_MAX}")
print(f"Columns: {len(df.columns):,}")
df.head(3)

In [ ]:
# -----------------------------
# Classification helper functions
# -----------------------------
# Note: all classifications here come from structured fields already present in the dataset, matched_* hit fields,
# and interpretable keyword rules in title/abstract/keywords/outcomes text.
# Each record enters only one category for each overview metric, so each panel can be drawn as a 100% stacked bar.

TEXT_COLS = ["project_title", "abstract_text", "keywords_raw", "outcomes_text", "include_reason", "review_flag"]


def row_text(row, cols=TEXT_COLS):
    """Combine multiple text fields from one row; used only for keyword rules and does not change raw data."""
    parts = []
    for col in cols:
        value = row.get(col, "")
        if pd.notna(value):
            parts.append(str(value))
    return " ".join(parts)


def split_terms(value):
    """Split matched_* fields on semicolons, pipes, enumeration commas, and similar separators into a set."""
    if pd.isna(value):
        return set()
    parts = re.split(r"[;；|,，、\s]+", str(value))
    return {part.strip() for part in parts if part.strip() and part.strip().lower() != "nan"}


def has_any(text, keywords):
    """Return whether the text contains any keyword."""
    return any(keyword in text for keyword in keywords)


# -----------------------------
# Single-record classifiers for the 10 overview metrics
# -----------------------------

def classify_data_source(row):
    """Data source: derived from country + source_system."""
    country = row.get("country")
    source = row.get("source_system")
    if pd.notna(country) and pd.notna(source):
        if country == "China" and "NSFC" in str(source):
            return "China / NSFC final DB"
        return f"{country} / {source}"
    if pd.notna(source):
        return str(source)
    return "Not reported"


def classify_method_family(row):
    """Method family: identify AI first, then urban sensing traces, remote sensing, GIS, and big-data/multisource methods."""
    terms = split_terms(row.get("matched_method_terms"))
    text = row_text(row)
    if terms & {"人工智能", "机器学习", "深度学习"} or has_any(
        text, ["神经网络", "随机森林", "梯度提升", "SHAP", "XGBoost", "LightGBM", "深度学习", "机器学习", "人工智能"]
    ):
        return "AI / machine learning"
    if terms & {"街景", "POI", "LBS", "手机信令", "轨迹", "LiDAR", "三维", "夜间灯光"}:
        return "Urban sensing traces"
    if terms & {"遥感", "高分"}:
        return "Remote sensing"
    if terms & {"GIS", "地理信息"}:
        return "GIS / spatial analysis"
    if terms & {"多源数据", "大数据"}:
        return "Big-data / multisource"
    return "Not classified"


def classify_built_environment_object(row):
    """Built-environment object: assign specific green/sponge, street/block, and building classes before land-use/urban-form or planning/city-region."""
    terms = split_terms(row.get("matched_object_terms"))
    if not terms:
        return "Not classified"
    if terms & {"绿地", "海绵"}:
        return "Green / sponge infrastructure"
    if terms & {"街道", "街区"}:
        return "Street / block"
    if terms & {"建筑"}:
        return "Building"
    if terms & {"建成环境", "城市形态", "土地利用"}:
        return "Land use / urban form"
    if terms & {"规划", "城市群"}:
        return "Planning / city-region"
    return "Not classified"


def classify_environmental_performance(row):
    """Environmental performance: classify using performance terms in matched_performance_terms; records without terms are Not reported."""
    terms = split_terms(row.get("matched_performance_terms"))
    if not terms:
        return "Not reported"
    if terms & {"碳", "能源"}:
        return "Carbon / energy"
    if terms & {"热岛", "热环境", "气候"}:
        return "Thermal / climate"
    if terms & {"空气污染", "暴露"}:
        return "Air / exposure"
    if terms & {"洪涝", "韧性"}:
        return "Flood / resilience"
    if terms & {"生态环境"}:
        return "Ecosystem quality"
    return "Not reported"


def classify_project_scale_place(row):
    """Project scale/place availability: check structured place fields first; if empty, report only scale cues from text."""
    structured_place_columns = ["knowledge_prod_place", "research_object_place", "beneficiary_city"]
    if any(pd.notna(row.get(col)) and str(row.get(col)).strip() not in {"", "nan"} for col in structured_place_columns):
        return "Structured place reported"
    text = row_text(row, ["project_title", "abstract_text", "keywords_raw", "outcomes_text"])
    if has_any(text, ["城市群", "都市圈", "区域", "流域", "跨区域", "省域", "京津冀", "长三角", "大湾区", "北部湾", "呼包鄂榆"]):
        return "Regional cue only"
    if has_any(text, ["街区", "街道", "社区", "居住区", "小区", "建筑", "校园", "学校", "学区"]):
        return "Neighbourhood/building cue"
    if has_any(text, ["城市", "城区", "市域", "都市"]):
        return "City cue only"
    return "Not reported"


def classify_agency_program(row):
    """Funding program type: use the project type before the semicolon in agency_program."""
    value = "" if pd.isna(row.get("agency_program")) else str(row.get("agency_program")).split(";")[0].strip()
    if "青年" in value:
        return "Young Scientists Fund"
    if "面上" in value:
        return "General Program"
    if "地区" in value:
        return "Regional Fund"
    if "重点" in value:
        return "Key Program"
    if "重大" in value:
        return "Major Research Plan"
    if "联合" in value:
        return "Joint Fund"
    return "Other / not classified"


def classify_funding_amount_availability(row):
    """Funding amount availability: amount is available when both amount_original and currency are present."""
    if pd.notna(row.get("amount_original")) and pd.notna(row.get("currency")):
        return "Amount reported"
    if pd.notna(row.get("amount_original")):
        return "Amount reported, currency missing"
    return "Not reported"


def classify_multisource_fusion(row):
    """Multisource fusion: identify explicit multisource/fusion wording first, then use the number of method terms as a weak proxy."""
    terms = split_terms(row.get("matched_method_terms"))
    text = row_text(row, ["project_title", "abstract_text", "keywords_raw", "outcomes_text", "matched_method_terms"])
    if "多源数据" in terms or has_any(text, ["多源", "数据融合", "多模态", "多传感器", "集成手机", "集成遥感", "大、小数据"]):
        return "Explicit fusion"
    if len(terms) >= 3:
        return "Multiple method cues"
    if len(terms) >= 1:
        return "Single method cue"
    return "Not classified"


def classify_review_priority(row):
    """Review priority: parse OK, no-explicit-performance-term, and broad-term cues from review_flag."""
    flag = "" if pd.isna(row.get("review_flag")) else str(row.get("review_flag"))
    if flag.strip() == "OK":
        return "OK"
    no_performance = "方法词+宽泛对象词且无明确环境绩效词" in flag
    broad_terms = "宽泛词命中较多" in flag
    if no_performance and broad_terms:
        return "No performance + broad"
    if no_performance:
        return "No performance"
    if broad_terms:
        return "Broad terms"
    if flag.startswith("PRIORITY_REVIEW"):
        return "Priority review"
    return "Not classified"


def classify_application_maturity_proxy(row):
    """Application/maturity proxy: use only visible cues in outcomes_text, such as outputs, uptake, tools, and data artifacts."""
    text = row_text(row, ["outcomes_text"])
    if not text.strip():
        return "Not reported"
    if re.search(r"采纳|采用|应用示范|示范应用|推广应用|实际应用|政策建议被|决策咨询|获.*奖|获得.*奖|成果转化|服务.*规划|服务.*治理|被.*部门", text):
        return "Policy/practice uptake"
    if re.search(r"软件著作权|专利|平台|系统|数据库|数据集|监测系统|评价系统|决策支持系统|工具", text):
        return "Tool/data artifact"
    if re.search(r"发表|论文|著作|会议|培养|研究报告|报告", text):
        return "Academic outputs"
    return "Not reported"


METRICS = [
    ("01_data_source", "Data source", classify_data_source),
    ("02_method_family", "Method family", classify_method_family),
    ("03_built_environment_object", "Built-environment object", classify_built_environment_object),
    ("04_environmental_performance", "Environmental performance", classify_environmental_performance),
    ("05_project_scale_place", "Project scale / place availability", classify_project_scale_place),
    ("06_agency_program", "Agency program", classify_agency_program),
    ("07_funding_amount_availability", "Funding amount availability", classify_funding_amount_availability),
    ("08_multisource_fusion", "Multisource fusion", classify_multisource_fusion),
    ("09_review_priority", "Review priority", classify_review_priority),
    ("10_application_maturity_proxy", "Application / maturity proxy", classify_application_maturity_proxy),
]

FIGURE_EXCLUDED_METRICS = {"01_data_source", "07_funding_amount_availability"}
FIGURE_METRICS = [metric for metric in METRICS if metric[0] not in FIGURE_EXCLUDED_METRICS]

# Display order for each metric; Not reported / Not classified appear last so they can be shown consistently in gray.
CATEGORY_ORDER = {
    "01_data_source": ["China / NSFC final DB", "Other", "Not reported"],
    "02_method_family": ["AI / machine learning", "Urban sensing traces", "Remote sensing", "GIS / spatial analysis", "Big-data / multisource", "Not classified"],
    "03_built_environment_object": ["Land use / urban form", "Planning / city-region", "Green / sponge infrastructure", "Building", "Street / block", "Not classified"],
    "04_environmental_performance": ["Thermal / climate", "Carbon / energy", "Flood / resilience", "Ecosystem quality", "Air / exposure", "Not reported"],
    "05_project_scale_place": ["Structured place reported", "Regional cue only", "City cue only", "Neighbourhood/building cue", "Not reported"],
    "06_agency_program": ["General Program", "Young Scientists Fund", "Regional Fund", "Key Program", "Major Research Plan", "Joint Fund", "Other / not classified"],
    "07_funding_amount_availability": ["Amount reported", "Amount reported, currency missing", "Not reported"],
    "08_multisource_fusion": ["Explicit fusion", "Multiple method cues", "Single method cue", "Not classified"],
    "09_review_priority": ["OK", "No performance", "Broad terms", "No performance + broad", "Priority review", "Not classified"],
    "10_application_maturity_proxy": ["Policy/practice uptake", "Tool/data artifact", "Academic outputs", "Not reported"],
}

RULE_SUMMARIES = {
    "01_data_source": "country + source_system",
    "02_method_family": "matched_method_terms plus method cues in title/abstract/keywords/outcomes; priority: AI, urban sensing, remote sensing, GIS, big-data/multisource",
    "03_built_environment_object": "matched_object_terms; priority: green/sponge, street/block, building, land-use/urban-form, planning/city-region",
    "04_environmental_performance": "matched_performance_terms; no term = Not reported",
    "05_project_scale_place": "structured place fields first; otherwise text scale cues only",
    "06_agency_program": "agency_program project type before semicolon",
    "07_funding_amount_availability": "amount_original and currency completeness",
    "08_multisource_fusion": "explicit multisource/fusion text first; otherwise number of matched_method_terms",
    "09_review_priority": "review_flag parsed into OK, no-performance, broad-term, or combined priority categories",
    "10_application_maturity_proxy": "outcomes_text cues for uptake, tool/data artifact, academic output, or Not reported",
}

In [ ]:
# -----------------------------
# Generate the metric table
# -----------------------------
# To make every small-multiple panel a 100% stacked bar, map each record to one category per metric,
# then aggregate counts and percentages by metric-category.

classified = df.copy()
for metric_id, metric_label, classifier in METRICS:
    classified[metric_id] = classified.apply(classifier, axis=1)

metric_rows = []
for metric_id, metric_label, _ in METRICS:
    counts = classified[metric_id].value_counts(dropna=False).to_dict()
    order = [category for category in CATEGORY_ORDER[metric_id] if category in counts]
    extra_categories = sorted(category for category in counts if category not in order)
    ordered_categories = order + extra_categories
    denominator = int(sum(counts.values()))
    for category in ordered_categories:
        count = int(counts[category])
        metric_rows.append({
            "metric_id": metric_id,
            "metric_label": metric_label,
            "category": category,
            "n": count,
            "denominator": denominator,
            "share": count / denominator if denominator else np.nan,
            "percent": round(100 * count / denominator, 1) if denominator else np.nan,
            "category_order": ordered_categories.index(category) + 1,
            "rule_summary": RULE_SUMMARIES[metric_id],
        })

metrics_table = pd.DataFrame(metric_rows)
bad_denominators = metrics_table.loc[metrics_table["denominator"] != N_RECORDS, ["metric_id", "denominator"]].drop_duplicates()
if not bad_denominators.empty:
    raise ValueError(f"Dashboard denominators must all equal N={N_RECORDS:,}: {bad_denominators.to_dict('records')}")

metrics_table.to_csv(TABLE_PATH, index=False, encoding="utf-8-sig")

print(f"Saved metrics table: {TABLE_PATH}")
metrics_table

In [ ]:
# -----------------------------
# Draw the 4 x 2 small-multiple 100% stacked-bar dashboard, excluding two constant 100% panels.
# -----------------------------
# All figure text is English; Not reported / Not classified categories use gray consistently.

# Publication-friendly matplotlib settings: white background, editable SVG/PDF text, light grid, and no default rainbow palette.
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.sans-serif": ["Times New Roman"],
    "font.monospace": ["Times New Roman"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "font.size": 7,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "legend.frameon": False,
})

sns.set_theme(
    style="whitegrid",
    rc={
        "font.family": "serif",
        "font.serif": ["Times New Roman"],
        "font.sans-serif": ["Times New Roman"],
        "font.monospace": ["Times New Roman"],
        "figure.facecolor": "#FFFFFF",
        "savefig.facecolor": "#FFFFFF",
        "axes.facecolor": "#FFFFFF",
        "axes.edgecolor": "#D9DEE2",
        "grid.color": "#E8ECEF",
        "grid.linewidth": 0.75,
        "axes.labelcolor": "#222222",
        "xtick.color": "#555555",
        "ytick.color": "#555555",
    },
)

TOKENS = {
    "ink": "#222222",
    "muted": "#555555",
    "grid": "#E8ECEF",
    "axis": "#D9DEE2",
    "grey_fill": "#D6D6D6",
    "grey_edge": "#B8B8B8",
}

SCALE_TICKS = [0, 25, 50, 75, 100]

# Fig7-derived low-saturation palette; gray is reserved for Not reported / Not classified.
# Within each panel, the highest non-missing share is light green and the lowest non-missing share is blue-gray.
HIGHEST_SHARE_COLOR = ("#C2D0B5", "#87A96B")  # light sage green
LOW_SHARE_COLOR = ("#9DBCC4", "#4B7F8C")      # blue-gray, moved to low-share categories
SECONDARY_COLORS = [
    ("#B7CEE2", "#6F9BBF"),  # soft blue, Fig7 remote-sensing family
    ("#E2C895", "#C9A66B"),  # ochre, Fig7 3D / LiDAR family
    ("#C7B9DC", "#9E8AC7"),  # muted purple, Fig7 exposure family
    ("#E6B98F", "#D39567"),  # muted orange, Fig7 thermal family
    ("#DFA99F", "#C66C5A"),  # muted red, Fig7 carbon-energy family
    ("#A9C7C7", "#76A6A6"),  # teal-gray, Fig7 comfort family
]


def is_missing_category(label):
    """Identify Not reported / Not classified categories that should use gray."""
    lower_label = str(label).lower()
    return "not reported" in lower_label or "not classified" in lower_label or "other / not classified" in lower_label


def assign_panel_colors(panel_data):
    """Assign colors within each panel by share: highest light green, lowest blue-gray, and missing/unclassified fixed gray."""
    colors = [None] * len(panel_data)
    valid_indices = []
    for idx, row in panel_data.iterrows():
        if is_missing_category(row["category"]):
            colors[idx] = (TOKENS["grey_fill"], TOKENS["grey_edge"])
        else:
            valid_indices.append(idx)

    if not valid_indices:
        return colors

    highest_idx = max(valid_indices, key=lambda idx: (float(panel_data.loc[idx, "percent"]), -idx))
    colors[highest_idx] = HIGHEST_SHARE_COLOR

    remaining_indices = [idx for idx in valid_indices if idx != highest_idx]
    if remaining_indices:
        lowest_idx = min(remaining_indices, key=lambda idx: (float(panel_data.loc[idx, "percent"]), idx))
        colors[lowest_idx] = LOW_SHARE_COLOR

    secondary_position = 0
    for idx in valid_indices:
        if colors[idx] is None:
            colors[idx] = SECONDARY_COLORS[secondary_position % len(SECONDARY_COLORS)]
            secondary_position += 1
    return colors


def wrap_legend_label(label, width=24):
    """Control the width of in-panel category descriptions to prevent text overlap."""
    return textwrap.fill(str(label), width=width, break_long_words=False)


def draw_metric_panel(ax, metric_id, metric_label, panel_data):
    """Draw one metric as a 100% horizontal stacked bar and place compact notes inside the panel."""
    panel_data = panel_data.sort_values("category_order").reset_index(drop=True)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.grid(False)
    ax.set_frame_on(False)
    ax.patch.set_edgecolor("none")
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.tick_params(axis="x", bottom=False, labelbottom=False, length=0)
    ax.tick_params(axis="y", left=False, labelleft=False, length=0)
    ax.set_title(metric_label, loc="left", fontsize=8.5, fontweight="semibold", color=TOKENS["ink"], pad=1)

    # Draw the single 100% stacked bar.
    left = 0.0
    bar_y = 0.87
    bar_h = 0.18
    scale_y = bar_y - bar_h / 2 - 0.045
    for tick in SCALE_TICKS:
        ha = "center"
        if tick == 0:
            ha = "left"
        elif tick == 100:
            ha = "right"
        ax.text(tick, scale_y, f"{tick}%", ha=ha, va="top", fontsize=5.3, color=TOKENS["muted"], clip_on=False)
    colors = assign_panel_colors(panel_data)
    for idx, row in panel_data.iterrows():
        percent = float(row["percent"])
        fill, edge = colors[idx]
        ax.barh(
            y=bar_y,
            width=percent,
            left=left,
            height=bar_h,
            color=fill,
            edgecolor="#FFFFFF",
            linewidth=1.0,
            zorder=3,
        )
        # Show only numeric percentages inside color blocks; put category names in the in-panel legend to avoid crowded bar labels.
        if percent >= 8:
            ax.text(left + percent / 2, bar_y, f"{percent:.1f}%", ha="center", va="center", fontsize=6.2, color=TOKENS["ink"])
        left += percent

    # Place two compact note columns inside each panel so all category names and percentages remain visible.
    n_categories = len(panel_data)
    rows_per_col = max(1, math.ceil(n_categories / 2))
    if rows_per_col <= 2:
        row_gap = 0.30
        start_y = 0.50
    elif rows_per_col == 3:
        row_gap = 0.20
        start_y = 0.55
    else:
        row_gap = 0.145
        start_y = 0.57
    for idx, row in panel_data.iterrows():
        column = idx // rows_per_col
        row_in_column = idx % rows_per_col
        x0 = 0.02 + column * 0.49
        y0 = start_y - row_in_column * row_gap
        fill, edge = colors[idx]
        ax.scatter([x0], [y0], transform=ax.transAxes, s=18, marker="s", color=fill, edgecolor=edge, linewidth=0.55, clip_on=False, zorder=5)
        legend_label = f"{wrap_legend_label(row['category'])}: {row['percent']:.1f}%"
        ax.text(x0 + 0.028, y0, legend_label, transform=ax.transAxes, ha="left", va="center", fontsize=5.85, color=TOKENS["muted"], linespacing=0.95)


fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(7.6, 5.0), sharex=False)
axes = axes.ravel()

for ax, (metric_id, metric_label, _) in zip(axes, FIGURE_METRICS):
    panel = metrics_table.loc[metrics_table["metric_id"] == metric_id].copy()
    draw_metric_panel(ax, metric_id, metric_label, panel)

fig.subplots_adjust(left=0.055, right=0.985, top=0.985, bottom=0.05, hspace=0.03, wspace=0.18)
fig.patch.set_facecolor("#FFFFFF")

# Export four figure formats: SVG/PDF preserve editable text, and TIFF/PNG use a high-resolution white background.
fig.savefig(f"{FIGURE_BASE}.svg", bbox_inches="tight", pad_inches=0.01, facecolor="#FFFFFF")
fig.savefig(f"{FIGURE_BASE}.pdf", bbox_inches="tight", pad_inches=0.01, facecolor="#FFFFFF")
fig.savefig(f"{FIGURE_BASE}.tiff", dpi=600, bbox_inches="tight", pad_inches=0.01, facecolor="#FFFFFF", pil_kwargs={"compression": "tiff_lzw"})
fig.savefig(f"{FIGURE_BASE}.png", dpi=450, bbox_inches="tight", pad_inches=0.01, facecolor="#FFFFFF")

print("Exported figure files:")
for suffix in ["svg", "pdf", "tiff", "png"]:
    path = Path(f"{FIGURE_BASE}.{suffix}")
    print(f"- {path} ({path.stat().st_size / 1024:.1f} KB)")

plt.show()

In [ ]:
# -----------------------------
# Save the draft results paragraph and classification notes
# -----------------------------
# The Markdown log supports reuse in manuscript/review writing; it is not raw data and does not overwrite other subagents' outputs.

summary_lookup = {
    metric_id: metrics_table.loc[metrics_table["metric_id"] == metric_id]
    .sort_values("category_order")
    .assign(label=lambda d: d["category"] + " (" + d["n"].astype(str) + ", " + d["percent"].map(lambda x: f"{x:.1f}%") + ")")
    ["label"]
    .tolist()
    for metric_id, _, _ in METRICS
}

lines = []
lines.append("# 05 Dashboard metrics and draft text")
lines.append("")
lines.append(f"- Source data: `{SOURCE_DATA_RELATIVE.as_posix()}`")
lines.append(f"- Records: {N_RECORDS:,}")
lines.append(f"- Award years: {AWARD_YEAR_MIN}-{AWARD_YEAR_MAX}")
lines.append(f"- Figure: `output/figures/Fig4_overview_dashboard.svg/.pdf/.tiff/.png`")
lines.append(f"- Metrics table: `output/tables/05_dashboard_metrics.csv`")
lines.append("")
lines.append("## Metric definitions")
for metric_id, metric_label, _ in METRICS:
    lines.append(f"- **{metric_label}**: {RULE_SUMMARIES[metric_id]}")
lines.append("")
lines.append("## Metric distributions")
for metric_id, metric_label, _ in METRICS:
    lines.append(f"- **{metric_label}**: " + "; ".join(summary_lookup[metric_id]))
lines.append("")
lines.append("## Draft results paragraph")
lines.append(
    f"The overview dashboard summarizes {N_RECORDS:,} China/NSFC final-database records "
    f"with award_year {AWARD_YEAR_MIN}-{AWARD_YEAR_MAX}. Funding-amount availability is tracked explicitly. "
    "Methodologically, the corpus is led by AI/machine learning, "
    "urban sensing traces, and remote sensing, while GIS/spatial analysis and big-data/multisource approaches "
    "form smaller but still visible groups. Built-environment objects are concentrated in land-use/urban-form, "
    "planning/city-region, green/sponge infrastructure, building, and street/block themes, with a residual "
    "not-classified group. Environmental performance cues are absent for a substantial subset, while thermal/climate "
    "and carbon/energy outcomes are the most visible reported performance dimensions. The structured place fields are "
    "largely unavailable, so project scale is represented by text-derived cues rather than curated study-place metadata."
)
lines.append("")
lines.append("## Classification uncertainty")
lines.append(f"- In the {N_RECORDS:,}-record master table, the three structured place fields (`knowledge_prod_place`, `research_object_place`, `beneficiary_city`) have no non-empty values; the scale/place panel therefore uses text-derived scale cues and should not be read as verified geocoded study-place metadata.")
lines.append("- Method, object, and performance families are single-label summaries. Records with multiple matched terms are assigned by an explicit priority rule, so secondary themes are intentionally compressed.")
lines.append("- Review-priority labels inherit the upstream `review_flag` logic. Broad terms such as planning, climate, carbon, energy, big data, and GIS may require manual screening for strict BAE measurement relevance.")
lines.append("- Application/maturity is a keyword proxy from `outcomes_text`; it indicates visible uptake, tool/data artifacts, or academic-output cues, not independently verified implementation maturity.")
lines.append("")

LOG_PATH.write_text("\n".join(lines), encoding="utf-8")
print(f"Saved text log: {LOG_PATH}")
print("\n".join(lines[:18]))